Imports

In [1]:
import sklearn
from sklearn.model_selection import train_test_split

In [2]:
import pandas as pd
import numpy as np

Load the real world dataset and display the structure

In [3]:
dataset = sklearn.datasets.load_diabetes( return_X_y=False, as_frame=False)

In [4]:
print(dataset.data[0])

[ 0.03807591  0.05068012  0.06169621  0.02187239 -0.0442235  -0.03482076
 -0.04340085 -0.00259226  0.01990749 -0.01764613]


In [5]:
print(dataset.target)

[151.  75. 141. 206. 135.  97. 138.  63. 110. 310. 101.  69. 179. 185.
 118. 171. 166. 144.  97. 168.  68.  49.  68. 245. 184. 202. 137.  85.
 131. 283. 129.  59. 341.  87.  65. 102. 265. 276. 252.  90. 100.  55.
  61.  92. 259.  53. 190. 142.  75. 142. 155. 225.  59. 104. 182. 128.
  52.  37. 170. 170.  61. 144.  52. 128.  71. 163. 150.  97. 160. 178.
  48. 270. 202. 111.  85.  42. 170. 200. 252. 113. 143.  51.  52. 210.
  65. 141.  55. 134.  42. 111.  98. 164.  48.  96.  90. 162. 150. 279.
  92.  83. 128. 102. 302. 198.  95.  53. 134. 144. 232.  81. 104.  59.
 246. 297. 258. 229. 275. 281. 179. 200. 200. 173. 180.  84. 121. 161.
  99. 109. 115. 268. 274. 158. 107.  83. 103. 272.  85. 280. 336. 281.
 118. 317. 235.  60. 174. 259. 178. 128.  96. 126. 288.  88. 292.  71.
 197. 186.  25.  84.  96. 195.  53. 217. 172. 131. 214.  59.  70. 220.
 268. 152.  47.  74. 295. 101. 151. 127. 237. 225.  81. 151. 107.  64.
 138. 185. 265. 101. 137. 143. 141.  79. 292. 178.  91. 116.  86. 122.
  72. 

Create a random permuntation so I can make sure that the model works and not only with single permuntation

In [6]:
X = dataset.data
Y = dataset.target

In [7]:
#set seed for this to be reproducible
np.random.seed(42)

n = X.shape[0]
perm = np.random.permutation(n)

X = X[perm]
Y = Y[perm]

In [8]:
print(X[0])

[ 0.04534098 -0.04464164 -0.00620595 -0.01599898  0.1250187   0.1251981
  0.019187    0.03430886  0.03243232 -0.0052198 ]


Create the training, test split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)


In [10]:
X_train.shape, y_train.shape


((353, 10), (353,))

In [11]:
X_test.shape, y_test.shape

((89, 10), (89,))

Add a bias (intercept) to the datapoints so the weights are not being updated that harshly

In [12]:
def add_bias(X):
    return np.c_[np.ones(X.shape[0]), X]

X_train_b = add_bias(X_train)
X_test_b = add_bias(X_test)

In [13]:
print(X_train_b[0])

[ 1.          0.06350368  0.05068012  0.08864151  0.0700723   0.02044629
  0.03751653 -0.05076412  0.07120998  0.02929656  0.07348023]


Determine simple functions

In [14]:
#Error

def calculcateError( label, predi):
  predictionError = ( predi - label ) ** 2
  ''' print( predictionError) '''
  return predictionError

def simpleError( label, predi):
  predictionError =  predi - label
  ''' print( predictionError) '''
  return predictionError

def sumError(y_pred, labels):
  totalModelError = 0
  for i in range(len(y_pred)):

    error  = calculcateError( labels[i], y_pred[i])
    totalModelError += error

  return totalModelError

def simpleSumError(y_pred, labels):
  totalModelError = 0
  for i in range(len(y_pred)):

    error  = simpleError( labels[i], y_pred[i])
    totalModelError += error

  return totalModelError


# crude and easy implementation of a prediction
def predict(X, w):
    return X @ w




''' sumError(y_pred) '''



' sumError(y_pred) '

Ridge

In [15]:
# ridge addon for weights
def getRidgePenalty(alpha, w):
    total = 0
    for i in range(len(w)):
      total += (w[i] ** 2)


    return alpha * total


def getRidgeGradient(alpha, w):
    return 2 * alpha * w

In [16]:
''' def gradientDescent(learningRate, X,y):

    m_gradient = 0
    b_gradient = 0
    n = len(X)

  return upWeights '''

def getFeatureBlame( X, y, y_pred):

    # figure out how each feature contributed to those errors
    return (X.T @ (y_pred - y))

In [23]:
# L2, ridge
def ridge(X, y):
  # Regulation strength or penalty strength
  alpha = 0.1
  learningRate = 0.1
  prev_loss = float("inf")

  iterations = 10000

  m, n = X.shape

  # Create weights
  w = np.zeros(n)

  for i in range(iterations):
    y_pred = predict(X, w)


    loss  = sumError(y_pred, Y)

    if loss >= prev_loss:
      learningRate *= 0.8
      ''' print("Reducing LR to", learningRate) '''
    prev_loss = loss

    gradient = (2 / m) *  getFeatureBlame(X, y ,y_pred) + getRidgeGradient(alpha, w)
    '''   print(gradient) '''
    # update weights
    w = w - learningRate * gradient
    # make the intercept to 0
    w[0] = 0
    ''' print(w ) '''
  return w

ridgeWeights = ridge(X_train_b, y_train)


testPredictions = predict(X_test_b, ridgeWeights)
print(sumError(testPredictions, y_test))

2117455.7353296187


In [18]:
X_train_b.shape

(353, 11)

In [30]:
for i in range(10):
  print("Error for " + str(i) + ": " + str(calculcateError(testPredictions[i], y_test[i])))

Error for 0: 20193.685188832023
Error for 1: 11439.630682296942
Error for 2: 74535.6970090694
Error for 3: 65574.988362394
Error for 4: 3883.155026686266
Error for 5: 39589.6475785403
Error for 6: 84318.03004005934
Error for 7: 5238.945489338365
Error for 8: 7992.618835241916
Error for 9: 12749.933292220254
